# Day 18 — Chi-square Test
> Testing independence between categorical variables.

## When to Use Chi-square?

- Both variables are **categorical**
- You have a **contingency table** (counts)
- Expected frequencies ≥ 5 in each cell

**H₀:** The two variables are independent (no association)
**H₁:** There is a statistically significant association

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style='whitegrid')
np.random.seed(42)

df = pd.read_csv('../data/device_purchase.csv')
print(df['device'].value_counts())
print(f"\nOverall purchase rate: {df['purchased'].mean():.3%}")


In [ ]:
# Build contingency table
ct = pd.crosstab(df['device'], df['purchased'], margins=True)
print("Contingency Table (counts):")
print(ct)

# Purchase rate per device
rate = pd.crosstab(df['device'], df['purchased'], normalize='index')
print("\nPurchase Rate by Device:")
print(rate.rename(columns={0:'Not Purchased', 1:'Purchased'}).round(4))


In [ ]:
# Chi-square test
observed = pd.crosstab(df['device'], df['purchased']).values
chi2, p_value, dof, expected = stats.chi2_contingency(observed)

print(f"Chi-square statistic : {chi2:.4f}")
print(f"Degrees of freedom   : {dof}")
print(f"p-value              : {p_value:.6f}")
print(f"Decision: {'Reject H₀ — device type affects purchasing' if p_value<0.05 else 'Fail to reject H₀'}")

# Cramér's V (effect size)
n = observed.sum()
cramers_v = np.sqrt(chi2 / (n * (min(observed.shape) - 1)))
print(f"Cramér's V = {cramers_v:.4f}  ({'weak' if cramers_v<0.1 else 'moderate' if cramers_v<0.3 else 'strong'} association)")


In [ ]:
# Visualize: observed vs expected
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Heatmap of observed
obs_df = pd.DataFrame(observed, index=['desktop','mobile','tablet'], columns=['Not Purchased','Purchased'])
sns.heatmap(obs_df, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Observed Frequencies')

# Purchase rate bar chart
rate_plot = rate.rename(columns={1:'Purchase Rate'})
rate_plot['Purchase Rate'].plot(kind='bar', ax=axes[1], color=['#4C72B0','#DD8452','#55A868'], edgecolor='white')
axes[1].set_title(f'Purchase Rate by Device  (χ²={chi2:.2f}, p={p_value:.4f})')
axes[1].set_ylabel('Rate')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].axhline(df['purchased'].mean(), color='red', linestyle='--', lw=1.5, label='Overall mean')
axes[1].legend()

plt.suptitle('Chi-square Test: Device Type vs Purchase Decision', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/04_chi_square.png', dpi=150)
plt.show()
